In [1]:
# ============================================================
# 1- Database design in phpMyAdmin -> SQL codes
# ============================================================

In [2]:
# ============================================================
# 2- Phone Number Normalization
# ============================================================

import re

_PERSIAN_DIGITS = str.maketrans(
    "۰۱۲۳۴۵۶۷۸۹",
    "0123456789"
)

_ARABIC_DIGITS = str.maketrans(
    "٠١٢٣٤٥٦٧٨٩",
    "0123456789"
)


def normalize_iran_phone(value) -> str:
    """
    Return an Iranian mobile number in local 10-digit form.

    Example:
        9121234567

    Accepted formats:
        9121234567
        09121234567
        +989121234567
        00989121234567

    Persian and Arabic-Indic digits are also accepted.
    """

    if value is None:
        raise ValueError("Phone number is empty")

    # Normalize Unicode digits first.
    s = str(value).strip()
    s = s.translate(_PERSIAN_DIGITS)
    s = s.translate(_ARABIC_DIGITS)

    # Reject obvious Excel numeric artifacts.
    if re.fullmatch(r"\d+\.0+", s):
        raise ValueError(
            f"Phone number appears to have been converted to a numeric "
            f"Excel value: {value!r}"
        )

    # Keep digits and an optional leading '+'.
    s = re.sub(r"[^\d+]", "", s)

    # Convert international Iranian formats to local format.
    if s.startswith("+98"):
        s = s[3:]
    elif s.startswith("0098"):
        s = s[4:]
    elif s.startswith("98") and len(s) == 12:
        s = s[2:]
    elif s.startswith("0"):
        s = s[1:]

    # Final Iranian mobile validation.
    if not re.fullmatch(r"9\d{9}", s):
        raise ValueError(
            f"Invalid Iranian mobile number: {value!r}"
        )

    return s

In [3]:
# ============================================================
# 3- Import Excel Data into the Database
# ============================================================
from pathlib import Path

import pandas as pd
import mysql.connector

EXCEL_FILE = Path(
    "/home/saeed/repositories/contacts/import_eitaa/contacts-1-44.xlsx"
)


def get_db_connection():
    return mysql.connector.connect(
        host="127.0.0.1",
        port=3306,
        user="reza",
        password="123",
        database="eitaa_import",
        charset="utf8mb4",
    )


def import_excel():
    df = pd.read_excel(
        EXCEL_FILE,
        dtype={
            "Full Name": "string",
            "Phone Number": "string",
        },
    )

    expected = {"Full Name", "Phone Number"}

    if set(df.columns) != expected:
        raise ValueError(
            f"Expected columns {expected}, got {set(df.columns)}"
        )

    conn = get_db_connection()

    try:
        cursor = conn.cursor()

        sql = """
            INSERT INTO contacts (
                full_name,
                phone_original,
                phone_normalized,
                status
            )
            VALUES (%s, %s, %s, 'pending')
            ON DUPLICATE KEY UPDATE
                full_name = VALUES(full_name),
                phone_original = VALUES(phone_original)
        """

        processed = 0
        invalid_phones = 0
        empty_names = 0

        for _, row in df.iterrows():

            # ------------------------------------------------
            # Full name
            # ------------------------------------------------
            if pd.isna(row["Full Name"]):
                empty_names += 1
                continue

            full_name = str(row["Full Name"]).strip()

            if not full_name:
                empty_names += 1
                continue

            # ------------------------------------------------
            # Phone number
            # ------------------------------------------------
            if pd.isna(row["Phone Number"]):
                invalid_phones += 1
                print("Invalid phone number: empty Excel cell")
                continue

            phone_original = str(row["Phone Number"]).strip()

            try:
                phone_normalized = normalize_iran_phone(
                    phone_original
                )
            except ValueError as exc:
                invalid_phones += 1
                print(
                    f"Invalid phone number "
                    f"{phone_original!r}: {exc}"
                )
                continue

            # ------------------------------------------------
            # Insert/update database
            # ------------------------------------------------
            cursor.execute(
                sql,
                (
                    full_name,
                    phone_original,
                    phone_normalized,
                ),
            )

            processed += 1

        conn.commit()

        # ----------------------------------------------------
        # Verify database state after import
        # ----------------------------------------------------
        cursor.execute(
            """
            SELECT
                COUNT(*) AS total,
                SUM(status = 'pending') AS pending,
                SUM(status = 'processing') AS processing,
                SUM(status = 'success') AS success,
                SUM(status = 'skipped_existing') AS skipped_existing,
                SUM(status = 'failed') AS failed
            FROM contacts
            """
        )

        stats = cursor.fetchone()

        print()
        print("========== IMPORT RESULT ==========")
        print(f"Excel rows processed : {processed}")
        print(f"Invalid phones       : {invalid_phones}")
        print(f"Empty names          : {empty_names}")
        print()
        print(f"DB total             : {stats[0]}")
        print(f"DB pending           : {stats[1]}")
        print(f"DB processing        : {stats[2]}")
        print(f"DB success           : {stats[3]}")
        print(f"DB skipped_existing  : {stats[4]}")
        print(f"DB failed            : {stats[5]}")
        print("====================================")

    except Exception:
        conn.rollback()
        raise

    finally:
        cursor.close()
        conn.close()


if __name__ == "__main__":
    import_excel()



========== IMPORT RESULT ==========
Excel rows processed : 44
Invalid phones       : 0
Empty names          : 0

DB total             : 44
DB pending           : 44
DB processing        : 0
DB success           : 0
DB skipped_existing  : 0
DB failed            : 0


In [4]:
# ============================================================
# 4- Chrome session persistence
# ============================================================
# Clean, isolated Selenium Chrome session

from pathlib import Path
from tempfile import mkdtemp
from typing import Optional
import shutil

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.webdriver import WebDriver

# ChromeDriver

CHROMEDRIVER = (
        Path.cwd()
        / "drivers"
        / "chromedriver"
        / "chromedriver"
)

# Create a completely new temporary Chrome User Data Directory

TEMP_CHROME_PROFILE = Path(
    mkdtemp(prefix="selenium-chrome-")
)

# Close previous Selenium session, if one exists

previous_driver: Optional[WebDriver] = globals().get("driver")

if previous_driver is not None:
    try:
        previous_driver.quit()
        print("\nPrevious Selenium driver closed.")
    except Exception as e:
        print("\nPrevious Selenium driver could not be closed:")
        print(repr(e))

# Chrome options

options = Options()

options.add_argument(
    f"--user-data-dir={TEMP_CHROME_PROFILE}"
)

options.add_argument("--start-maximized")

#  Explicit manual ChromeDriver
#  Selenium Manager is not used for driver discovery.


service = Service(
    executable_path=str(CHROMEDRIVER)
)

# Create new isolated Chrome session


driver: WebDriver = webdriver.Chrome(
    service=service,
    options=options,
)

driver.set_page_load_timeout(60)


In [29]:
# ============================================================
# 5- Close Selenium and remove temporary Chrome profile
# ============================================================

if "driver" in globals() and driver is not None:
    try:
        driver.quit()
        print("Selenium Chrome session closed.")
    finally:
        driver = None

if "TEMP_CHROME_PROFILE" in globals():
    shutil.rmtree(
        TEMP_CHROME_PROFILE,
        ignore_errors=True
    )

    print(
        "Temporary Selenium Chrome profile removed."
    )

Selenium Chrome session closed.
Temporary Selenium Chrome profile removed.


In [27]:
# ============================================================
# 6- Selenium Selectors
# ============================================================

from selenium.webdriver.common.by import By


class Selectors:
    # 1. Hamburger menu
    HAMBURGER_MENU = (
        By.CSS_SELECTOR,
        ".btn-icon.btn-menu-toggle.rp.sidebar-tools-button.is-visible"
    )

    # 2. Contacts menu item
    CONTACTS_BUTTON = (
        By.XPATH,
        "//div[contains(@class, 'btn-menu-item') and .//span[normalize-space()='Contacts']]"
    )

    # 3. "+" button inside Contacts
    ADD_CONTACT_BUTTON = (
        By.XPATH,
        "//button[contains(@class, 'btn-circle') "
        "and contains(@class, 'btn-corner') "
        "and .//span[@data-icon='add']]"
    )

    # 4. First name field
    FIRST_NAME = (
        By.CSS_SELECTOR,
        'div.input-field:first-child > div.input-field-input[contenteditable="true"]'
    )

    # 5. Phone number field
    PHONE = (
        By.CSS_SELECTOR,
        'div.input-field-phone > '
        'div.input-field-input[contenteditable="true"]'
    )

    # 6. Add/Save button
    SAVE_BUTTON = (
        By.CSS_SELECTOR,
        ".popup-create-contact button.btn-primary.btn-color-primary"
    )

    # 7. Toast shown when the phone number is not registered on Eitaa
    UNREGISTERED_TOAST = (
        By.CSS_SELECTOR,
        "div.toast.is-visible"
    )

    # 8. Close Add Contact window ( × button )
    CLOSE_CONTACT_WINDOW = (
        By.CSS_SELECTOR,
        ".popup-create-contact button.btn-icon.popup-close"
    )

In [28]:
# ============================================================
# TEST - Section 6: Selenium Selectors
# ============================================================

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import time


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

WAIT_TIMEOUT = 10

# Test number that produces the Eitaa "not registered" toast
TEST_UNREGISTERED_PHONE = "9111111111"

wait = WebDriverWait(driver, WAIT_TIMEOUT)


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def verify_visible(name, locator):
    """
    Verify that a selector finds a visible element.
    """
    try:
        element = wait.until(
            EC.visibility_of_element_located(locator)
        )

        print(f"✓ {name}")
        return element

    except TimeoutException:
        raise AssertionError(
            f"✗ {name} FAILED\n"
            f"  Locator: {locator}"
        )


def click_element(name, locator):
    """
    Wait for an element to be clickable and click it.
    """
    try:
        element = wait.until(
            EC.element_to_be_clickable(locator)
        )

        element.click()
        print(f"✓ {name} - clicked")
        return element

    except TimeoutException:
        raise AssertionError(
            f"✗ {name} FAILED - element was not clickable\n"
            f"  Locator: {locator}"
        )


# ============================================================
# 1. Verify HAMBURGER_MENU
# ============================================================

print("=" * 70)
print("SECTION 6 - SELENIUM SELECTOR TEST")
print("=" * 70)

print("\n[1] HAMBURGER_MENU")
print("-" * 70)

verify_visible(
    "HAMBURGER_MENU",
    Selectors.HAMBURGER_MENU
)


# ============================================================
# 2. Open hamburger menu
# ============================================================

print("\n[2] Open Hamburger Menu")
print("-" * 70)

click_element(
    "HAMBURGER_MENU",
    Selectors.HAMBURGER_MENU
)


# ============================================================
# 3. Verify CONTACTS_BUTTON
# ============================================================

print("\n[3] CONTACTS_BUTTON")
print("-" * 70)

verify_visible(
    "CONTACTS_BUTTON",
    Selectors.CONTACTS_BUTTON
)


# ============================================================
# 4. Open Contacts
# ============================================================

print("\n[4] Open Contacts")
print("-" * 70)

click_element(
    "CONTACTS_BUTTON",
    Selectors.CONTACTS_BUTTON
)


# ============================================================
# 5. Verify ADD_CONTACT_BUTTON
# ============================================================

print("\n[5] ADD_CONTACT_BUTTON")
print("-" * 70)

verify_visible(
    "ADD_CONTACT_BUTTON",
    Selectors.ADD_CONTACT_BUTTON
)


# ============================================================
# 6. Open Add Contact dialog
# ============================================================

print("\n[6] Open Add Contact Dialog")
print("-" * 70)

click_element(
    "ADD_CONTACT_BUTTON",
    Selectors.ADD_CONTACT_BUTTON
)


# ============================================================
# 7. Verify FIRST_NAME
# ============================================================

print("\n[7] FIRST_NAME")
print("-" * 70)

first_name = verify_visible(
    "FIRST_NAME",
    Selectors.FIRST_NAME
)


# ============================================================
# 8. Verify PHONE
# ============================================================

print("\n[8] PHONE")
print("-" * 70)

phone = verify_visible(
    "PHONE",
    Selectors.PHONE
)


# ============================================================
# 9. Verify SAVE_BUTTON
# ============================================================

print("\n[9] SAVE_BUTTON")
print("-" * 70)

verify_visible(
    "SAVE_BUTTON",
    Selectors.SAVE_BUTTON
)


# ============================================================
# 10. Verify CLOSE_CONTACT_WINDOW
# ============================================================

print("\n[10] CLOSE_CONTACT_WINDOW")
print("-" * 70)

verify_visible(
    "CLOSE_CONTACT_WINDOW",
    Selectors.CLOSE_CONTACT_WINDOW
)


# ============================================================
# 11. Enter test phone number
# ============================================================

print("\n[11] Enter Test Phone Number")
print("-" * 70)

phone.click()
phone.send_keys(TEST_UNREGISTERED_PHONE)

print(
    f"✓ PHONE accepted test number: "
    f"{TEST_UNREGISTERED_PHONE}"
)


# ============================================================
# 12. Enter test first name
# ============================================================

print("\n[12] Enter Test First Name")
print("-" * 70)

first_name.click()
first_name.send_keys("SelectorTest")

print("✓ FIRST_NAME accepted test text")


# ============================================================
# 13. Click SAVE
# ============================================================

print("\n[13] SAVE_BUTTON")
print("-" * 70)

click_element(
    "SAVE_BUTTON",
    Selectors.SAVE_BUTTON
)


# ============================================================
# 14. Detect transient UNREGISTERED_TOAST
#
# IMPORTANT:
# The toast exists only briefly, so we poll the DOM
# frequently instead of using a long normal WebDriverWait.
# ============================================================

print("\n[14] UNREGISTERED_TOAST")
print("-" * 70)

toast_found = False

toast_locator = Selectors.UNREGISTERED_TOAST

toast_deadline = time.monotonic() + 5

while time.monotonic() < toast_deadline:

    try:
        toast_elements = driver.find_elements(
            *toast_locator
        )

        for toast in toast_elements:

            if toast.is_displayed():

                toast_found = True

                print("✓ UNREGISTERED_TOAST appeared")

                try:
                    print(
                        f"  Toast text: "
                        f"{toast.text!r}"
                    )
                except Exception:
                    pass

                break

        if toast_found:
            break

    except Exception:
        pass

    time.sleep(0.05)


if not toast_found:
    raise AssertionError(
        "✗ UNREGISTERED_TOAST FAILED\n"
        "The toast did not appear within the 5-second "
        "monitoring window."
    )


# ============================================================
# 15. CLOSE TOAST
#
# The same CLOSE_CONTACT_WINDOW selector is used.
# The first click dismisses the toast.
# ============================================================

print("\n[15] Close Toast")
print("-" * 70)

click_element(
    "CLOSE_CONTACT_WINDOW - first click (dismiss toast)",
    Selectors.CLOSE_CONTACT_WINDOW
)

print("✓ Toast dismissed")


# ------------------------------------------------------------
# Give Eitaa a very short moment to update the UI.
# ------------------------------------------------------------

time.sleep(0.2)


# ============================================================
# 16. CLOSE ADD CONTACT WINDOW
#
# The same selector must be clicked a second time.
# ============================================================

print("\n[16] Close Add Contact Window")
print("-" * 70)

click_element(
    "CLOSE_CONTACT_WINDOW - second click (close window)",
    Selectors.CLOSE_CONTACT_WINDOW
)

print("✓ Add Contact window close action executed")


# ============================================================
# 17. Verify Add Contact window is actually closed
# ============================================================

print("\n[17] Verify Add Contact Window Closed")
print("-" * 70)

try:
    WebDriverWait(driver, 5).until(
        EC.invisibility_of_element_located(
            Selectors.CLOSE_CONTACT_WINDOW
        )
    )

    print("✓ Add Contact window is closed")

except TimeoutException:
    raise AssertionError(
        "✗ Add Contact window is still visible after "
        "the second CLOSE_CONTACT_WINDOW click."
    )


# ============================================================
# 18. Final result
# ============================================================

print("\n" + "=" * 70)
print("SECTION 6 TEST PASSED")
print("=" * 70)

print("""
All Section 6 Selenium selectors were successfully verified:

✓ HAMBURGER_MENU
✓ CONTACTS_BUTTON
✓ ADD_CONTACT_BUTTON
✓ FIRST_NAME
✓ PHONE
✓ SAVE_BUTTON
✓ UNREGISTERED_TOAST
✓ CLOSE_CONTACT_WINDOW

The transient UNREGISTERED_TOAST was detected successfully.

Cleanup completed:

✓ First CLOSE_CONTACT_WINDOW click dismissed the toast
✓ Second CLOSE_CONTACT_WINDOW click closed the Add Contact window
✓ Add Contact window disappearance was verified
""")


SECTION 6 - SELENIUM SELECTOR TEST

[1] HAMBURGER_MENU
----------------------------------------------------------------------
✓ HAMBURGER_MENU

[2] Open Hamburger Menu
----------------------------------------------------------------------
✓ HAMBURGER_MENU - clicked

[3] CONTACTS_BUTTON
----------------------------------------------------------------------
✓ CONTACTS_BUTTON

[4] Open Contacts
----------------------------------------------------------------------
✓ CONTACTS_BUTTON - clicked

[5] ADD_CONTACT_BUTTON
----------------------------------------------------------------------
✓ ADD_CONTACT_BUTTON

[6] Open Add Contact Dialog
----------------------------------------------------------------------
✓ ADD_CONTACT_BUTTON - clicked

[7] FIRST_NAME
----------------------------------------------------------------------
✓ FIRST_NAME

[8] PHONE
----------------------------------------------------------------------
✓ PHONE

[9] SAVE_BUTTON
----------------------------------------------------

In [5]:
# ============================================================
# 7- Reliable waits
# ============================================================

# Use explicit waits instead of time.sleep().
#
# The purpose of this section is to provide reusable,
# reliable synchronization helpers for the Selenium code
# used in later sections.
#
# These helpers are designed for situations such as:
#   - waiting for Chrome/Eitaa elements to appear
#   - waiting for an element to become visible
#   - waiting for an element to become clickable
#   - waiting for an element/popup to disappear
#   - waiting for a specific Selenium condition
#   - handling temporary/stale DOM elements
#
# ============================================================

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    StaleElementReferenceException,
    NoSuchElementException,
    ElementClickInterceptedException,
)

# ------------------------------------------------------------
# General configuration
# ------------------------------------------------------------

WAIT_SECONDS = 30


# ------------------------------------------------------------
# 1. Create a WebDriverWait object
# ------------------------------------------------------------

def get_wait(driver, timeout=WAIT_SECONDS):
    """
    Return a WebDriverWait instance.

    Parameters
    ----------
    driver : WebDriver
        Active Selenium WebDriver.

    timeout : int | float
        Maximum number of seconds to wait.

    Returns
    -------
    WebDriverWait
    """
    return WebDriverWait(
        driver,
        timeout,
        poll_frequency=0.2,
        ignored_exceptions=(
            NoSuchElementException,
            StaleElementReferenceException,
        ),
    )


# ------------------------------------------------------------
# 2. Wait until an element exists in the DOM
# ------------------------------------------------------------

def wait_present(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is present in the DOM.

    The element does not necessarily have to be visible.

    Useful when:
        - the element exists but is not yet displayed
        - another wait will handle visibility/clickability
    """
    return get_wait(driver, timeout).until(
        EC.presence_of_element_located(locator)
    )


# ------------------------------------------------------------
# 3. Wait until an element is visible
# ------------------------------------------------------------

def wait_visible(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is present and visible.

    Returns
    -------
    WebElement
    """
    return get_wait(driver, timeout).until(
        EC.visibility_of_element_located(locator)
    )


# ------------------------------------------------------------
# 4. Wait until an element is clickable
# ------------------------------------------------------------

def wait_clickable(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is visible and enabled/clickable.

    This should normally be used before clicking buttons,
    links, menu items, etc.
    """
    return get_wait(driver, timeout).until(
        EC.element_to_be_clickable(locator)
    )


# ------------------------------------------------------------
# 5. Wait until an element disappears
# ------------------------------------------------------------

def wait_invisible(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is no longer visible.

    This is particularly useful for:
        - closing popups
        - waiting for dialogs to disappear
        - waiting for loading overlays to disappear

    Note:
        Selenium's invisibility_of_element_located() also
        succeeds if the element is removed from the DOM.
    """
    return get_wait(driver, timeout).until(
        EC.invisibility_of_element_located(locator)
    )


# ------------------------------------------------------------
# 6. Wait until an element disappears from the DOM
# ------------------------------------------------------------

def wait_stale(driver, element, timeout=WAIT_SECONDS):
    """
    Wait until a previously obtained WebElement becomes stale.

    Useful when an existing element is replaced/removed after
    an operation such as saving a form.
    """
    return get_wait(driver, timeout).until(
        EC.staleness_of(element)
    )


# ------------------------------------------------------------
# 7. Wait for a locator to contain specific text
# ------------------------------------------------------------

def wait_text(driver, locator, text, timeout=WAIT_SECONDS):
    """
    Wait until an element contains the specified text.
    """
    return get_wait(driver, timeout).until(
        EC.text_to_be_present_in_element(
            locator,
            text,
        )
    )


# ------------------------------------------------------------
# 8. Wait for a specific attribute value
# ------------------------------------------------------------

def wait_attribute(
        driver,
        locator,
        attribute,
        value,
        timeout=WAIT_SECONDS,
):
    """
    Wait until an element's attribute contains the specified value.

    Parameters
    ----------
    driver : WebDriver
        Active Selenium WebDriver.

    locator : tuple
        Selenium locator.

    attribute : str
        Name of the HTML attribute to inspect.

    value : str
        Value that must be present in the attribute.

    timeout : int | float
        Maximum number of seconds to wait.

    Returns
    -------
    bool
        True when the attribute contains the specified value.
    """

    def attribute_contains(driver):
        try:
            element = driver.find_element(*locator)
            attribute_value = element.get_attribute(attribute)

            if attribute_value is None:
                return False

            return value in attribute_value

        except (
                NoSuchElementException,
                StaleElementReferenceException,
        ):
            return False

    return get_wait(driver, timeout).until(
        attribute_contains
    )


# ------------------------------------------------------------
# 9. Wait until an element is selected
# ------------------------------------------------------------

def wait_selected(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until a selectable element is selected.
    """
    return get_wait(driver, timeout).until(
        EC.element_located_to_be_selected(locator)
    )


# ------------------------------------------------------------
# 10. Wait for a custom condition
# ------------------------------------------------------------

def wait_until(
        driver,
        condition,
        timeout=WAIT_SECONDS,
):
    """
    Wait until a custom Selenium condition returns a truthy
    value.

    The condition receives the driver.

    Example:

        wait_until(
            driver,
            lambda d: some_condition(d)
        )

    This is useful when Selenium's built-in expected
    conditions are not sufficient.
    """
    return get_wait(driver, timeout).until(condition)


# ------------------------------------------------------------
# 11. Safe click helper
# ------------------------------------------------------------

def wait_and_click(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is clickable and then click it.

    Returns
    -------
    WebElement
        The element that was clicked.
    """
    element = wait_clickable(
        driver,
        locator,
        timeout,
    )

    element.click()

    return element


# ------------------------------------------------------------
# 12. Wait for a popup to close
# ------------------------------------------------------------

def wait_popup_closed(
        driver,
        popup_locator,
        timeout=WAIT_SECONDS,
):
    """
    Wait until a popup/dialog is no longer visible.

    This is a convenience wrapper around
    wait_invisible().

    It is intended for situations such as the Eitaa
    Add Contact popup.
    """
    return wait_invisible(
        driver,
        popup_locator,
        timeout,
    )


# ------------------------------------------------------------
# 13. Wait for an element and then enter text
# ------------------------------------------------------------

def wait_and_type(
        driver,
        locator,
        text,
        timeout=WAIT_SECONDS,
        clear_first=True,
):
    """
    Wait for an input field to become visible and type text.

    Parameters
    ----------
    driver : WebDriver
        Active Selenium WebDriver.

    locator : tuple
        Selenium locator, for example:
            (By.CSS_SELECTOR, "...")

    text : str
        Text to enter.

    timeout : int | float
        Maximum number of seconds to wait.

    clear_first : bool
        If True, clear the input before typing.

    Returns
    -------
    WebElement
        The input element.
    """
    element = wait_visible(
        driver,
        locator,
        timeout,
    )

    if clear_first:
        element.clear()

    element.send_keys(text)

    return element

In [ ]:
DELETE ME!
# ============================================================
# TEST — Section 7: Reliable waits
# ============================================================
#
# This test verifies all reusable wait helpers defined in
# Section 7.
#
# The test uses a temporary local HTML page, so it does NOT:
#   - modify Eitaa
#   - add any contacts
#   - modify the MySQL database
#   - require an Eitaa login
#
# It uses the existing Selenium driver from Section 4.
#
# ============================================================

from selenium.webdriver.common.by import By
from selenium.common.exceptions import TimeoutException

print("=" * 60)
print("SECTION 7 TEST — RELIABLE WAITS")
print("=" * 60)

# ------------------------------------------------------------
# 1. Verify required Section 7 objects exist
# ------------------------------------------------------------

required_functions = [
    "get_wait",
    "wait_present",
    "wait_visible",
    "wait_clickable",
    "wait_invisible",
    "wait_stale",
    "wait_text",
    "wait_attribute",
    "wait_selected",
    "wait_until",
    "wait_and_click",
    "wait_popup_closed",
    "wait_and_type",
]

missing_functions = [
    name
    for name in required_functions
    if name not in globals() or not callable(globals()[name])
]

if missing_functions:
    raise AssertionError(
        "Missing Section 7 functions: "
        + ", ".join(missing_functions)
    )

print("✓ All Section 7 functions are available")

# ------------------------------------------------------------
# 2. Verify Selenium driver exists
# ------------------------------------------------------------

if "driver" not in globals() or driver is None:
    raise RuntimeError(
        "Selenium driver was not found. "
        "Run Section 4 before running this test."
    )

print("✓ Selenium driver is available")

# ------------------------------------------------------------
# 3. Create a temporary test page
# ------------------------------------------------------------

test_html = """
<!DOCTYPE html>
<html>
<head>
    <title>Section 7 Wait Test</title>
</head>

<body>

    <div id="present-element">
        Present element
    </div>

    <div id="visible-element">
        Visible element
    </div>

    <button id="click-button" type="button">
        Click me
    </button>

    <div id="text-element">
        Waiting for test text
    </div>

    <div id="attribute-element" class="test-element">
        Attribute test
    </div>

    <input
        id="text-input"
        type="text"
        value="old value"
    >

    <select id="select-element">
        <option value="one">One</option>
        <option value="two" selected>Two</option>
    </select>

    <div id="popup">
        Test popup
    </div>

    <div id="stale-element">
        This element will become stale
    </div>

    <div id="custom-element">
        Custom condition
    </div>

    <script>

        // Make the text condition available shortly after load.
        setTimeout(function () {
            document.getElementById("text-element").textContent =
                "Expected test text";
        }, 300);

        // Change the class attribute shortly after load.
        setTimeout(function () {
            document.getElementById("attribute-element")
                .className = "test-element active";
        }, 300);

        // Hide the popup shortly after load.
        setTimeout(function () {
            document.getElementById("popup").style.display = "none";
        }, 300);

        // Make the custom condition true shortly after load.
        setTimeout(function () {
            document.getElementById("custom-element")
                .setAttribute("data-ready", "true");
        }, 300);

    </script>

</body>
</html>
"""

# Load the HTML directly in Chrome.
driver.get(
    "data:text/html;charset=utf-8,"
    + test_html
)

# ------------------------------------------------------------
# 4. Define test locators
# ------------------------------------------------------------

PRESENT = (By.ID, "present-element")
VISIBLE = (By.ID, "visible-element")
CLICK_BUTTON = (By.ID, "click-button")
TEXT_ELEMENT = (By.ID, "text-element")
ATTRIBUTE_ELEMENT = (By.ID, "attribute-element")
TEXT_INPUT = (By.ID, "text-input")
SELECT_ELEMENT = (By.ID, "select-element")
POPUP = (By.ID, "popup")
STALE_ELEMENT = (By.ID, "stale-element")
CUSTOM_ELEMENT = (By.ID, "custom-element")

# ------------------------------------------------------------
# 5. Test get_wait()
# ------------------------------------------------------------

wait_object = get_wait(driver, timeout=5)

assert isinstance(
    wait_object,
    type(get_wait(driver, timeout=1))
)

print("✓ get_wait()")

# ------------------------------------------------------------
# 6. Test wait_present()
# ------------------------------------------------------------

element = wait_present(
    driver,
    PRESENT,
    timeout=5,
)

assert element is not None
assert element.get_attribute("id") == "present-element"

print("✓ wait_present()")

# ------------------------------------------------------------
# 7. Test wait_visible()
# ------------------------------------------------------------

element = wait_visible(
    driver,
    VISIBLE,
    timeout=5,
)

assert element.is_displayed()

print("✓ wait_visible()")

# ------------------------------------------------------------
# 8. Test wait_clickable()
# ------------------------------------------------------------

element = wait_clickable(
    driver,
    CLICK_BUTTON,
    timeout=5,
)

assert element.is_displayed()
assert element.is_enabled()

print("✓ wait_clickable()")

# ------------------------------------------------------------
# 9. Test wait_and_click()
# ------------------------------------------------------------

clicked_element = wait_and_click(
    driver,
    CLICK_BUTTON,
    timeout=5,
)

assert clicked_element.get_attribute("id") == "click-button"

print("✓ wait_and_click()")

# ------------------------------------------------------------
# 10. Test wait_text()
# ------------------------------------------------------------

wait_text(
    driver,
    TEXT_ELEMENT,
    "Expected test text",
    timeout=5,
)

actual_text = driver.find_element(
    *TEXT_ELEMENT
).text

assert "Expected test text" in actual_text

print("✓ wait_text()")

# ------------------------------------------------------------
# 11. Test wait_attribute()
# ------------------------------------------------------------

wait_attribute(
    driver,
    ATTRIBUTE_ELEMENT,
    "class",
    "active",
    timeout=5,
)

attribute_value = driver.find_element(
    *ATTRIBUTE_ELEMENT
).get_attribute("class")

assert "active" in attribute_value

print("✓ wait_attribute()")

# ------------------------------------------------------------
# 12. Test wait_selected()
# ------------------------------------------------------------

selected = wait_selected(
    driver,
    (By.CSS_SELECTOR, "#select-element option:checked"),
    timeout=5,
)

assert selected is not False

print("✓ wait_selected()")

# ------------------------------------------------------------
# 13. Test wait_and_type()
# ------------------------------------------------------------

typed_element = wait_and_type(
    driver,
    TEXT_INPUT,
    "Section 7 test",
    timeout=5,
)

assert typed_element.get_attribute("value") == "Section 7 test"

print("✓ wait_and_type()")

# ------------------------------------------------------------
# 14. Test wait_invisible()
# ------------------------------------------------------------

wait_invisible(
    driver,
    POPUP,
    timeout=5,
)

popup = driver.find_element(*POPUP)

assert not popup.is_displayed()

print("✓ wait_invisible()")

# ------------------------------------------------------------
# 15. Test wait_popup_closed()
# ------------------------------------------------------------

# The popup is already invisible, so this verifies that
# wait_popup_closed() correctly delegates to wait_invisible().

wait_popup_closed(
    driver,
    POPUP,
    timeout=5,
)

print("✓ wait_popup_closed()")

# ------------------------------------------------------------
# 16. Test wait_stale()
# ------------------------------------------------------------

# Create a fresh element specifically for this test.
driver.execute_script("""
    const element = document.createElement("div");
    element.id = "stale-test-element";
    element.textContent = "This element will become stale";
    document.body.appendChild(element);
""")

STALE_TEST_ELEMENT = (
    By.ID,
    "stale-test-element",
)

# Obtain the WebElement before it is removed.
stale_element = wait_present(
    driver,
    STALE_TEST_ELEMENT,
    timeout=5,
)

# Remove the element after a short delay.
driver.execute_script("""
    const element =
        document.getElementById("stale-test-element");

    setTimeout(function () {
        if (element) {
            element.remove();
        }
    }, 300);
""")

# Wait until the previously obtained WebElement becomes stale.
wait_stale(
    driver,
    stale_element,
    timeout=5,
)

print("✓ wait_stale()")

# ------------------------------------------------------------
# 17. Test wait_until()
# ------------------------------------------------------------

wait_until(
    driver,
    lambda d: d.find_element(
        *CUSTOM_ELEMENT
    ).get_attribute("data-ready") == "true",
    timeout=5,
)

assert driver.find_element(
    *CUSTOM_ELEMENT
).get_attribute("data-ready") == "true"

print("✓ wait_until()")

# ------------------------------------------------------------
# 18. Final result
# ------------------------------------------------------------

print()
print("=" * 60)
print("SECTION 7 TEST PASSED")
print("=" * 60)
print()
print("All Section 7 reliable-wait helpers passed successfully.")
print("The Selenium driver is still available for later sections.")

In [7]:
# ============================================================
# 8- Add Contact to Eitaa
# ============================================================

from datetime import datetime
from selenium.webdriver.common.by import By
from selenium.common.exceptions import (
    TimeoutException,
    StaleElementReferenceException,
)

# ------------------------------------------------------------
# Eitaa result message
# ------------------------------------------------------------

UNREGISTERED_MESSAGE = (
    "The person with this phone number is not registered on Eitaa yet."
)


# ------------------------------------------------------------
# Database status update helper
# ------------------------------------------------------------

def update_contact_status(
        phone_normalized,
        status,
        error_message=None,
        increment_attempt=False,
):
    """
    Update the corresponding contact in the database.

    Parameters
    ----------
    phone_normalized : str
        Normalized Iranian mobile number.

    status : str
        One of:
            pending
            processing
            success
            skipped
            failed

    error_message : str | None
        Error information for failed contacts.

    increment_attempt : bool
        If True, increase attempts by one.
    """

    conn = get_db_connection()

    try:
        cursor = conn.cursor()

        if increment_attempt:
            sql = """
                UPDATE contacts
                SET
                    status = %s,
                    attempts = attempts + 1,
                    error_message = %s,
                    completed_at = NULL
                WHERE phone_normalized = %s
            """

            cursor.execute(
                sql,
                (
                    status,
                    error_message,
                    phone_normalized,
                ),
            )

        else:
            sql = """
                UPDATE contacts
                SET
                    status = %s,
                    error_message = %s
                WHERE phone_normalized = %s
            """

            cursor.execute(
                sql,
                (
                    status,
                    error_message,
                    phone_normalized,
                ),
            )

        conn.commit()

    except Exception:
        conn.rollback()
        raise

    finally:
        cursor.close()
        conn.close()


# ------------------------------------------------------------
# Mark contact as processing
# ------------------------------------------------------------

def mark_contact_processing(phone_normalized):
    """
    Mark a contact as processing and record its start time.

    The attempt counter is increased here because every call to
    add_contact() represents one actual Eitaa attempt.
    """

    conn = get_db_connection()

    try:
        cursor = conn.cursor()

        cursor.execute(
            """
            UPDATE contacts
            SET
                status = 'processing',
                attempts = attempts + 1,
                error_message = NULL,
                started_at = NOW(),
                completed_at = NULL
            WHERE phone_normalized = %s
            """,
            (phone_normalized,),
        )

        if cursor.rowcount != 1:
            raise ValueError(
                f"Contact not found in database: {phone_normalized}"
            )

        conn.commit()

    except Exception:
        conn.rollback()
        raise

    finally:
        cursor.close()
        conn.close()


# ------------------------------------------------------------
# Mark final contact result
# ------------------------------------------------------------

def mark_contact_result(
        phone_normalized,
        status,
        error_message=None,
):
    """
    Store the final result of the current Eitaa attempt.

    Final states handled here:
        success
        skipped
        failed
    """

    conn = get_db_connection()

    try:
        cursor = conn.cursor()

        cursor.execute(
            """
            UPDATE contacts
            SET
                status = %s,
                error_message = %s,
                completed_at = NOW()
            WHERE phone_normalized = %s
            """,
            (
                status,
                error_message,
                phone_normalized,
            ),
        )

        if cursor.rowcount != 1:
            raise ValueError(
                f"Contact not found in database: {phone_normalized}"
            )

        conn.commit()

    except Exception:
        conn.rollback()
        raise

    finally:
        cursor.close()
        conn.close()


# ------------------------------------------------------------
# Detect the Eitaa "not registered" toast
# ------------------------------------------------------------

def is_unregistered_contact(driver):
    """
    Return True if Eitaa displays the exact unregistered-contact
    message.

    The toast does not have its own close button. The Add Contact
    window's close button must therefore be clicked twice.
    """

    try:
        elements = driver.find_elements(
            By.XPATH,
            (
                "//*[contains("
                "normalize-space(text()), "
                f"{UNREGISTERED_MESSAGE!r}"
                ")]"
            ),
        )

        for element in elements:
            try:
                if element.is_displayed():
                    return True
            except StaleElementReferenceException:
                continue

    except Exception:
        pass

    return False


# ------------------------------------------------------------
# Close Add Contact window
# ------------------------------------------------------------

def close_add_contact_window(driver):
    """
    Click the Add Contact window's × button.

    For an unregistered contact this function must be called
    twice:

        1. First click dismisses the toast.
        2. Second click closes the Add Contact window.
    """

    close_button = wait_clickable(
        driver,
        Selectors.CLOSE_CONTACT_WINDOW,
    )

    close_button.click()


# ------------------------------------------------------------
# Add one contact to Eitaa
# ------------------------------------------------------------

def add_contact(driver, full_name, phone_normalized):
    """
    Add exactly one contact to Eitaa.

    Parameters
    ----------
    driver : selenium.webdriver.WebDriver
        Active Selenium WebDriver.

    full_name : str
        Contact's full name.

    phone_normalized : str
        Normalized Iranian mobile number, e.g. 9121234567.

    Returns
    -------
    str
        One of:

            "success"
            "skipped"
            "failed"

    Behavior
    --------
    Registered contact:
        success -> no retry.

    Unregistered contact:
        skipped -> no retry.

    Selenium/Eitaa error:
        failed -> Section 10 decides whether another attempt
        should be made.
    """

    # --------------------------------------------------------
    # Validate basic input
    # --------------------------------------------------------

    if not full_name or not str(full_name).strip():
        raise ValueError("Contact name is empty.")

    if not phone_normalized:
        raise ValueError("Normalized phone number is empty.")

    full_name = str(full_name).strip()
    phone_normalized = str(phone_normalized).strip()

    # --------------------------------------------------------
    # Mark this contact as processing.
    #
    # attempts is incremented here.
    # Section 10 will decide whether a failed contact should
    # be attempted again.
    # --------------------------------------------------------

    mark_contact_processing(phone_normalized)

    try:

        # ====================================================
        # 1. Make sure we are on the Contacts page
        # ====================================================

        contacts_containers = driver.find_elements(
            By.CSS_SELECTOR,
            "#contacts-container",
        )

        if contacts_containers:
            contacts_container = contacts_containers[0]
        else:
            contacts_container = None

        contacts_visible = (
                contacts_container is not None
                and "active"
                in contacts_container.get_attribute("class").split()
        )

        if not contacts_visible:
            wait_and_click(
                driver,
                Selectors.HAMBURGER_MENU,
            )

            wait_and_click(
                driver,
                Selectors.CONTACTS_BUTTON,
            )

        # ====================================================
        # 2. Open Add Contact
        # ====================================================

        add_button = wait_clickable(
            driver,
            Selectors.ADD_CONTACT_BUTTON,
        )

        add_button.click()

        # ====================================================
        # 3. Fill first-name/full-name field
        # ====================================================

        wait_and_type(
            driver,
            Selectors.FIRST_NAME,
            full_name,
            clear_first=True,
        )

        # ====================================================
        # 4. Fill phone-number field
        # ====================================================

        wait_and_type(
            driver,
            Selectors.PHONE,
            phone_normalized,
            clear_first=True,
        )

        # ====================================================
        # 5. Click Save
        # ====================================================

        save_button = wait_clickable(
            driver,
            Selectors.SAVE_BUTTON,
        )

        save_button.click()

        # ====================================================
        # 6. Determine Eitaa's result
        #
        # For an unregistered number Eitaa leaves the
        # Add Contact window open and displays the toast.
        # Therefore we check for the exact toast before
        # waiting for the popup to disappear.
        # ====================================================

        try:
            get_wait(driver, WAIT_SECONDS).until(
                lambda d: (
                        is_unregistered_contact(d)
                        or not d.find_elements(
                    By.CSS_SELECTOR,
                    ".popup-create-contact.active",
                )
                )
            )

        except TimeoutException:
            raise RuntimeError(
                "Timed out while waiting for Eitaa to report "
                "the result of the Add Contact operation."
            )

        # ====================================================
        # 7. Unregistered number
        # ====================================================

        if is_unregistered_contact(driver):
            # ------------------------------------------------
            # First click:
            # dismiss the unregistered-number toast.
            # ------------------------------------------------

            close_add_contact_window(driver)

            # ------------------------------------------------
            # Second click:
            # close the Add Contact window itself.
            # ------------------------------------------------

            close_add_contact_window(driver)

            # ------------------------------------------------
            # Confirm that the Add Contact popup is closed.
            # ------------------------------------------------

            wait_popup_closed(
                driver,
                (
                    By.CSS_SELECTOR,
                    ".popup-create-contact.active",
                ),
            )

            # ------------------------------------------------
            # Store "skipped".
            #
            # This is NOT a failure and must NOT be retried.
            # ------------------------------------------------

            mark_contact_result(
                phone_normalized,
                status="skipped",
                error_message=None,
            )

            print(
                f"[SKIPPED] {full_name} - "
                f"{phone_normalized} "
                f"(not registered on Eitaa)"
            )

            return "skipped"

        # ====================================================
        # 8. Successful contact
        #
        # At this point the unregistered toast did not appear
        # and the Add Contact popup has closed.
        # ====================================================

        wait_popup_closed(
            driver,
            (
                By.CSS_SELECTOR,
                ".popup-create-contact.active",
            ),
        )

        # ----------------------------------------------------
        # Store success.
        # ----------------------------------------------------

        mark_contact_result(
            phone_normalized,
            status="success",
            error_message=None,
        )

        print(
            f"[SUCCESS] {full_name} - {phone_normalized}"
        )

        return "success"

    # ========================================================
    # 9. Any unexpected Selenium/Eitaa error
    # ========================================================

    except Exception as exc:

        error_message = (
            f"{type(exc).__name__}: {exc}"
        )

        # ----------------------------------------------------
        # Best-effort cleanup.
        #
        # If the Add Contact window is still open, try to
        # close it so Section 10 can continue with the next
        # attempt/contact.
        # ----------------------------------------------------

        try:
            popup = driver.find_elements(
                By.CSS_SELECTOR,
                ".popup-create-contact.active",
            )

            if popup:
                close_add_contact_window(driver)

                try:
                    wait_popup_closed(
                        driver,
                        (
                            By.CSS_SELECTOR,
                            ".popup-create-contact.active",
                        ),
                        timeout=5,
                    )
                except Exception:
                    pass

        except Exception:
            pass

        # ----------------------------------------------------
        # Store failed.
        #
        # The attempt was already incremented when the contact
        # was marked "processing".
        #
        # Section 10 is responsible for deciding whether this
        # failed contact should be retried.
        # ----------------------------------------------------

        try:
            mark_contact_result(
                phone_normalized,
                status="failed",
                error_message=error_message,
            )
        except Exception as db_exc:
            raise RuntimeError(
                "The Eitaa operation failed and the database "
                "could not be updated. "
                f"Eitaa error: {error_message}; "
                f"Database error: {db_exc}"
            ) from db_exc

        print(
            f"[FAILED] {full_name} - {phone_normalized}"
        )
        print(
            f"         {error_message}"
        )

        return "failed"


In [8]:
# ============================================================
# 9- Fetch Next Contact from Database
# ============================================================

def get_next_contact():
    """
    Fetch the next contact that should be processed.

    Processing order:
        1. New/pending contacts.
        2. Failed contacts with fewer than 3 attempts.

    Returns
    -------
    dict | None
        A dictionary containing:

            id
            full_name
            phone_original
            phone_normalized
            status
            attempts

        Returns None when there are no more contacts to process.

    Notes
    -----
    Section 9 only fetches a contact.

    It does NOT:
        - open Eitaa
        - call add_contact()
        - retry contacts
        - change the contact status

    Section 10 is responsible for calling add_contact() and
    handling the returned result.
    """

    conn = get_db_connection()

    try:
        cursor = conn.cursor(dictionary=True)

        cursor.execute(
            """
            SELECT
                id,
                full_name,
                phone_original,
                phone_normalized,
                status,
                attempts
            FROM contacts
            WHERE
                status = 'pending'
                OR (
                    status = 'failed'
                    AND attempts < 3
                )
            ORDER BY
                CASE
                    WHEN status = 'pending' THEN 0
                    WHEN status = 'failed' THEN 1
                END,
                id ASC
            LIMIT 1
            """
        )

        contact = cursor.fetchone()

        return contact

    finally:
        cursor.close()
        conn.close()


# ------------------------------------------------------------
# Convenience function for displaying the next contact
# ------------------------------------------------------------

def fetch_next_contact():
    """
    Fetch and display the next contact to be processed.

    This is a small wrapper around get_next_contact() and is
    useful for testing Section 9 manually before Section 10
    is implemented.

    Returns
    -------
    dict | None
        The next contact, or None if no contact remains.
    """

    contact = get_next_contact()

    if contact is None:
        print("No contacts remain to be processed.")
        return None

    print()
    print("========== NEXT CONTACT ==========")
    print(f"ID               : {contact['id']}")
    print(f"Full name        : {contact['full_name']}")
    print(f"Phone original   : {contact['phone_original']}")
    print(f"Phone normalized : {contact['phone_normalized']}")
    print(f"Status           : {contact['status']}")
    print(f"Attempts         : {contact['attempts']}")
    print("==================================")

    return contact


In [9]:
# ============================================================
# 10- Main Processing Loop
# ============================================================

# Section 10 is responsible for:
#
#   1. Fetching the next contact using Section 9.
#   2. Processing exactly one contact using Section 8.
#   3. Allowing Section 8 to record:
#        - success
#        - skipped
#        - failed
#   4. Continuing to the next contact.
#   5. Retrying failed contacts up to 3 total attempts.
#
# Section 8 performs one actual Eitaa attempt.
# Section 9 determines which contact should be processed next.
#
# No additional Selenium wait mechanism is introduced here.


MAX_ATTEMPTS = 3


def process_all_contacts(driver):
    """
    Process all eligible contacts from the database.

    Processing rules
    ----------------
    success:
        The contact was added successfully.
        No retry.

    skipped:
        The phone number is not registered on Eitaa.
        No retry.

    failed:
        The Eitaa operation failed.
        Retry while the contact has fewer than MAX_ATTEMPTS
        total attempts.

    The actual attempt counter is maintained by Section 8
    through mark_contact_processing().
    """

    processed_count = 0
    success_count = 0
    skipped_count = 0
    failed_count = 0

    print()
    print("============================================================")
    print("STARTING EITAA CONTACT IMPORT")
    print("============================================================")
    print()

    while True:

        # --------------------------------------------------------
        # 1. Fetch the next contact
        # --------------------------------------------------------
        contact = get_next_contact()

        # Section 9 returns None when no eligible contacts remain.
        if contact is None:
            break

        # --------------------------------------------------------
        # 2. Extract contact information
        # --------------------------------------------------------
        contact_id = contact["id"]
        full_name = contact["full_name"]
        phone_normalized = contact["phone_normalized"]
        current_status = contact["status"]
        previous_attempts = contact["attempts"]

        # Section 8 increments attempts when it starts the
        # actual Eitaa operation.
        current_attempt = previous_attempts + 1

        print()
        print("------------------------------------------------------------")
        print(
            f"Processing contact ID={contact_id} | "
            f"{full_name} | {phone_normalized}"
        )
        print(
            f"Status before attempt : {current_status}"
        )
        print(
            f"Attempt               : "
            f"{current_attempt}/{MAX_ATTEMPTS}"
        )
        print("------------------------------------------------------------")

        # --------------------------------------------------------
        # 3. Process exactly one contact
        # --------------------------------------------------------
        #
        # Section 8:
        #
        #   - marks the contact as processing
        #   - increments attempts
        #   - performs the Eitaa operation
        #   - stores success/skipped/failed
        #
        # Section 10 does NOT duplicate that database logic.

        try:
            result = add_contact(
                driver=driver,
                full_name=full_name,
                phone_normalized=phone_normalized,
            )

        except Exception as exc:
            # Normally add_contact() handles its own Eitaa/Selenium
            # exceptions and returns "failed".
            #
            # This catch protects the main loop from an unexpected
            # exception escaping Section 8.

            result = "failed"

            print()
            print(
                f"[MAIN LOOP ERROR] "
                f"{type(exc).__name__}: {exc}"
            )

        # --------------------------------------------------------
        # 4. Handle the result
        # --------------------------------------------------------

        if result == "success":

            success_count += 1
            processed_count += 1

            print(
                f"[COMPLETED] Contact ID={contact_id} "
                f"added successfully."
            )

            # Do not retry.
            # Continue directly to the next contact.
            continue

        elif result == "skipped":

            skipped_count += 1
            processed_count += 1

            print(
                f"[COMPLETED] Contact ID={contact_id} "
                f"skipped because the phone number is not "
                f"registered on Eitaa."
            )

            # Do not retry.
            # Continue directly to the next contact.
            continue

        elif result == "failed":

            failed_count += 1

            # ----------------------------------------------------
            # Section 8 already incremented the attempt counter.
            #
            # The current attempt number is therefore:
            #
            #     previous_attempts + 1
            # ----------------------------------------------------

            if current_attempt < MAX_ATTEMPTS:

                print(
                    f"[RETRY] Contact ID={contact_id} failed "
                    f"on attempt {current_attempt}/{MAX_ATTEMPTS}."
                )

                print(
                    f"        It will be retried after the "
                    f"remaining pending contacts are processed."
                )

                # Do not retry here directly.
                #
                # Instead, continue the loop. Section 9 will
                # determine the next eligible contact.
                continue

            else:

                processed_count += 1

                print(
                    f"[FAILED] Contact ID={contact_id} failed "
                    f"after {MAX_ATTEMPTS} attempts."
                )

                print(
                    "         No more retries will be performed."
                )

                # Continue to the next contact.
                continue

        else:
            # ----------------------------------------------------
            # Unexpected result from Section 8
            # ----------------------------------------------------
            #
            # This should never happen because Section 8 is
            # documented to return only:
            #
            #     success
            #     skipped
            #     failed
            #
            # Raise the error rather than silently continuing.

            raise ValueError(
                f"Unexpected result from add_contact(): "
                f"{result!r}"
            )

    # ============================================================
    # Final summary
    # ============================================================

    print()
    print("============================================================")
    print("EITAA CONTACT IMPORT FINISHED")
    print("============================================================")
    print(f"Successfully added : {success_count}")
    print(f"Skipped            : {skipped_count}")
    print(f"Final failed       : {failed_count}")
    print("============================================================")